In [1]:
import re
import joblib
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer

preprocessed_data_dir = Path("..\src\data\preprocessed_data")
preprocessed_data_dir.mkdir(parents=True, exist_ok=True)

<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Ekaterina\AppData\Local\Temp\ipykernel_20788\835237433.py:7: SyntaxWarning: invalid escape sequence '\s'
  preprocessed_data_dir = Path("..\src\data\preprocessed_data")


In [2]:
df_movies = pd.read_csv(r'..\src\data\movies.csv')
df_ratings = pd.read_csv(r'..\src\data\ratings.csv')
df_tags = pd.read_csv(r'..\src\data\tags.csv')
df_links = pd.read_csv(r'..\src\data\links.csv')

пропуски пот годам, поскольку их немного, заполним вручную, по обработке пропусков по жанрам подумать, функциии сделать воспроизводимыми, закинуть py файлик в src.

In [3]:
def extract_year(title):
    if not isinstance(title, str):
        return None

    matches = re.findall(r'\((.*?)\)', title)
        
    if not matches:
        return None

    last_skobka = matches[-1]

    year_ = re.search(r'(\d{4})', last_skobka)

    if year_:
        year = int(year_.group(1))
        return year
    
    return None


def era(year):
    if pd.isna(year):
        return 'Unknown'
    elif year < 1980:
        return 'Old'
    elif year < 1990:
        return "1980's"
    elif year < 2000:
        return "1990's"
    elif year < 2010:
        return "2000's"
    else:
        return "2020's"

def split_genres(genre_row):

    if isinstance(genre_row, list):
        return genre_row
    
    if isinstance(genre_row, str):
        return genre_row.split('|')
    
    return None


In [4]:
df_movies['year'] = df_movies['title'].apply(extract_year)
df_movies['era'] = df_movies['year'].apply(era)
df_movies['genres'] = df_movies['genres'].apply(split_genres)
df_movies.head()

,movieId,title,genres,year,era
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1995.0,1990's
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",1995.0,1990's
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",1995.0,1990's
4,5,Father of the Bride Part II (1995),[Comedy],1995.0,1990's


In [5]:
df_users = df_ratings.groupby('userId').agg(
    avg_user_rating=('rating', 'mean'),
    user_rating_count=('rating', 'count')
).reset_index()
df_users.head()

,userId,avg_user_rating,user_rating_count
0,1,4.366379,232
1,2,3.948276,29
2,3,2.435897,39
3,4,3.555556,216
4,5,3.636364,44


In [6]:
mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(df_movies['genres'])
genre_columns = mlb.classes_
genres_enc_df = pd.DataFrame(
    genres_encoded, 
    columns=[f'genre_{g}' for g in genre_columns],
    index=df_movies.index
)
genres_enc_df

,genre_(no genres listed),genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9737,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9738,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9739,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
9740,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [7]:
df_movies_featd = pd.concat([df_movies, genres_enc_df], axis=1)
df_movies_featd.head()

,movieId,title,genres,year,era,genre_(no genres listed),genre_Action,genre_Adventure,genre_Animation,genre_Children,...,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1995.0,1990's,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",1995.0,1990's,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",1995.0,1990's,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),[Comedy],1995.0,1990's,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
df_films_ratings = df_ratings.groupby('movieId').agg(
    avg_film_rating=('rating', 'mean'),
    film_rating_count=('rating', 'count')
).reset_index()
df_films_ratings.head()

,movieId,avg_film_rating,film_rating_count
0,1,3.920930,215
1,2,3.431818,110
2,3,3.259615,52
3,4,2.357143,7
4,5,3.071429,49


In [9]:
df_user_stats = df_ratings.groupby('userId').agg(
    avg_user_rating=('rating', 'mean'),
    user_rating_count=('rating', 'count')
).reset_index()
df_user_stats.head()
df_user_film_rait = df_ratings.merge(df_user_stats, 'inner', on='userId')
df_user_film_rait.head()

,userId,movieId,rating,timestamp,avg_user_rating,user_rating_count
0,1,1,4.0,964982703,4.366379,232
1,1,3,4.0,964981247,4.366379,232
2,1,6,4.0,964982224,4.366379,232
3,1,47,5.0,964983815,4.366379,232
4,1,50,5.0,964982931,4.366379,232


In [10]:
df_movies_join_rait = df_movies_featd.merge(df_films_ratings, "left", on='movieId')
df_movies_join_rait.head()

,movieId,title,genres,year,era,genre_(no genres listed),genre_Action,genre_Adventure,genre_Animation,genre_Children,...,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western,avg_film_rating,film_rating_count
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's,0,0,1,1,1,...,0,0,0,0,0,0,0,0,3.920930,215.0
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1995.0,1990's,0,0,1,0,1,...,0,0,0,0,0,0,0,0,3.431818,110.0
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",1995.0,1990's,0,0,0,0,0,...,0,0,0,1,0,0,0,0,3.259615,52.0
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",1995.0,1990's,0,0,0,0,0,...,0,0,0,1,0,0,0,0,2.357143,7.0
4,5,Father of the Bride Part II (1995),[Comedy],1995.0,1990's,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3.071429,49.0


In [11]:
df_final = df_movies_join_rait.merge(df_user_film_rait, 'inner', on='movieId')
df_final.head()

,movieId,title,genres,year,era,genre_(no genres listed),genre_Action,genre_Adventure,genre_Animation,genre_Children,...,genre_Thriller,genre_War,genre_Western,avg_film_rating,film_rating_count,userId,rating,timestamp,avg_user_rating,user_rating_count
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's,0,0,1,1,1,...,0,0,0,3.92093,215.0,1,4.0,964982703,4.366379,232
1,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's,0,0,1,1,1,...,0,0,0,3.92093,215.0,5,4.0,847434962,3.636364,44
2,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's,0,0,1,1,1,...,0,0,0,3.92093,215.0,7,4.5,1106635946,3.230263,152
3,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's,0,0,1,1,1,...,0,0,0,3.92093,215.0,15,2.5,1510577970,3.448148,135
4,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's,0,0,1,1,1,...,0,0,0,3.92093,215.0,17,4.5,1305696483,4.209524,105


In [12]:
df_final.isna().sum()

movieId                      0
title                        0
genres                       0
year                        17
era                          0
genre_(no genres listed)     0
genre_Action                 0
genre_Adventure              0
genre_Animation              0
genre_Children               0
genre_Comedy                 0
genre_Crime                  0
genre_Documentary            0
genre_Drama                  0
genre_Fantasy                0
genre_Film-Noir              0
genre_Horror                 0
genre_IMAX                   0
genre_Musical                0
genre_Mystery                0
genre_Romance                0
genre_Sci-Fi                 0
genre_Thriller               0
genre_War                    0
genre_Western                0
avg_film_rating              0
film_rating_count            0
userId                       0
rating                       0
timestamp                    0
avg_user_rating              0
user_rating_count            0
dtype: i

In [13]:
missing_years = df_final[df_final['year'].isna()][['movieId', 'title']]
print(missing_years)

        movieId                                              title
81292     40697                                          Babylon 5
81293     40697                                          Babylon 5
99325    140956                                   Ready Player One
99326    140956                                   Ready Player One
99327    140956                                   Ready Player One
99328    140956                                   Ready Player One
99459    143410                                         Hyena Road
99526    147250  The Adventures of Sherlock Holmes and Doctor W...
99617    149334                                  Nocturnal Animals
99799    156605                                           Paterson
100051   162414                                          Moonlight
100269   167570                                             The OA
100426   171495                                             Cosmos
100427   171495                                             Co

In [14]:
year_fixes = {
    40697: 1993,   # Babylon 5
    140956: 2018,  # Ready Player One
    143410: 2015,  # Hyena Road
    147250: 1980,  # The Adventures of Sherlock Holmes and Doctor Watson
    149334: 2016,  # Nocturnal Animals
    156605: 2016,  # Paterson
    162414: 2016,  # Moonlight
    167570: 2016,  # The OA
    171495: 2019,  # Cosmos
    171631: 2017,  # Maria Bamford: Old Baby
    171891: 2017,  # Generation Iron 2
    176601: 2011   # Black Mirror
}

for movie_id, year in year_fixes.items():
    df_final.loc[df_final['movieId'] == movie_id, 'year'] = year

df_final['era'] = df_final['year'].apply(era)
df_final.isna().sum()

movieId                     0
title                       0
genres                      0
year                        0
era                         0
genre_(no genres listed)    0
genre_Action                0
genre_Adventure             0
genre_Animation             0
genre_Children              0
genre_Comedy                0
genre_Crime                 0
genre_Documentary           0
genre_Drama                 0
genre_Fantasy               0
genre_Film-Noir             0
genre_Horror                0
genre_IMAX                  0
genre_Musical               0
genre_Mystery               0
genre_Romance               0
genre_Sci-Fi                0
genre_Thriller              0
genre_War                   0
genre_Western               0
avg_film_rating             0
film_rating_count           0
userId                      0
rating                      0
timestamp                   0
avg_user_rating             0
user_rating_count           0
dtype: int64

Уже для финального датасета обработали пропуски, закодировали жанры для дальнейшего построения моделей рекомендаций. Осталось только сохранить финальный датасет (еще таам время преобразовать)

In [15]:
svd_data = df_final[['userId', 'movieId', 'rating']]
content_data = df_final[['movieId'] + genres_enc_df.columns.tolist() + ['year']]
hybrid_data = df_final[['userId', 'movieId', 'rating', 'avg_user_rating', 'user_rating_count', 
                  'avg_film_rating', 'film_rating_count'] + genres_enc_df.columns.tolist()]  
#просто выделяю для себя, какие пока модели планируются и какие признаки им будут подаваться. мб потом убрать 

In [16]:
df_final['timestamp'] = pd.to_datetime(df_ratings['timestamp'], unit='s')
df_final.head()

,movieId,title,genres,year,era,genre_(no genres listed),genre_Action,genre_Adventure,genre_Animation,genre_Children,...,genre_Thriller,genre_War,genre_Western,avg_film_rating,film_rating_count,userId,rating,timestamp,avg_user_rating,user_rating_count
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's,0,0,1,1,1,...,0,0,0,3.92093,215.0,1,4.0,2000-07-30 18:45:03,4.366379,232
1,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's,0,0,1,1,1,...,0,0,0,3.92093,215.0,5,4.0,2000-07-30 18:20:47,3.636364,44
2,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's,0,0,1,1,1,...,0,0,0,3.92093,215.0,7,4.5,2000-07-30 18:37:04,3.230263,152
3,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's,0,0,1,1,1,...,0,0,0,3.92093,215.0,15,2.5,2000-07-30 19:03:35,3.448148,135
4,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's,0,0,1,1,1,...,0,0,0,3.92093,215.0,17,4.5,2000-07-30 18:48:51,4.209524,105


In [18]:
#теперь осталось только все корректно сохранить 
df_final.to_parquet(preprocessed_data_dir / 'full_data.parquet', engine='pyarrow', index=False)
joblib.dump(mlb, preprocessed_data_dir / 'mlb_genres.pkl')

#на всякий случай привычный csv сохраню, если не получится с parquet
df_final.to_csv(preprocessed_data_dir / 'full_data_csv_version.csv',  index=False)

В данном ноутбуке был сформирован общий финальный датасет, обработаны пропуски (вручную, тк их было мало в данных), построены новые признаки для дальнейшей работы